<a href="https://colab.research.google.com/github/bosywahab818-a11y/flyrank-ml-internship-bouthina/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bosywahab818-a11y/flyrank-ml-internship-bouthina/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles/folders:")
print(os.listdir())

Current folder:
/content

Files/folders:
['.config', 'sample_data']


In [2]:
!git clone "https://github.com/bosywahab818-a11y/flyrank-ml-internship-bouthina.git"

Cloning into 'flyrank-ml-internship-bouthina'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 128 (delta 41), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.86 MiB | 4.99 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [3]:
%cd /content/flyrank-ml-internship-bouthina

/content/flyrank-ml-internship-bouthina


In [4]:
import os

print(os.listdir("data"))
print(os.listdir("data/raw"))

['raw']
['content_refresh_anonymized.csv']


In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
print("Signal 1 - days_since_last_update")
print(df["days_since_last_update"].describe())

print("\nSignal 2 - ctr")
print(df["ctr"].describe())

Signal 1 - days_since_last_update
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Signal 2 - ctr
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


In [7]:
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, float("inf")],
    labels=["0-30 days", "31-90 days", "90+ days"]
)

signal_1 = (
    df.groupby("stale_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

signal_1

,stale_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,0-30 days,20480,4199.614062,13.727393,0.609021
1,31-90 days,175,6506.748571,9.685714,0.117543
2,90+ days,9345,7369.097057,21.411236,0.302696


In [8]:
df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-0.001, 0.05, 0.5, float("inf")],
    labels=["Low CTR", "Medium CTR", "High CTR"]
)

signal_2 = (
    df.groupby("ctr_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_position=("avg_position", "mean"),
          avg_clicks=("clicks_90d", "mean")
      )
      .reset_index()
)

signal_2

,ctr_bucket,n,avg_impressions,avg_position,avg_clicks
0,Low CTR,14419,1413.256675,19.276330,0.333172
1,Medium CTR,11432,9341.692880,14.557768,20.615465
2,High CTR,4149,6950.833454,11.063292,58.433357


### Signal checks

**Signal 1 — Freshness / staleness:** MIXED  
Older content shows different performance patterns, but the relationship is not consistently monotonic. The 90+ day bucket does not perform consistently worse than recently updated content, so staleness is useful as a directional signal but is not strong enough by itself.

**Signal 2 — CTR:** CONFIRMED  
CTR shows a clear relationship with search performance. Higher-CTR content has better average position and substantially more clicks in the observed data, so CTR is a useful signal for prioritization.

### Baseline rule

I will prioritize content with stronger opportunities for improvement using CTR and freshness as decision-support signals. The score will combine a low-CTR opportunity signal with a staleness signal, without using future-window or label-derived fields.

### Reason codes

- `LOW_CTR`: content has low CTR and may benefit from CTR-focused review.
- `STALE_CONTENT`: content has not been updated recently and may benefit from a freshness review.
- `LOW_CTR_AND_STALE`: both signals are present, giving higher priority.
- `REVIEW`: does not strongly match the other conditions but is still included in the ranked queue.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
# Create a copy for the baseline queue
queue = df.copy()

# Signal 1: low CTR opportunity
queue["low_ctr_signal"] = (queue["ctr"] < 0.5).astype(int)

# Signal 2: stale content
queue["stale_signal"] = (queue["days_since_last_update"] > 90).astype(int)

# Baseline score
queue["score"] = (
    queue["low_ctr_signal"] * 2
    + queue["stale_signal"] * 1
)

# Reason code
queue["reason_code"] = "REVIEW"

queue.loc[
    (queue["low_ctr_signal"] == 1) &
    (queue["stale_signal"] == 1),
    "reason_code"
] = "LOW_CTR_AND_STALE"

queue.loc[
    (queue["low_ctr_signal"] == 1) &
    (queue["stale_signal"] == 0),
    "reason_code"
] = "LOW_CTR"

queue.loc[
    (queue["low_ctr_signal"] == 0) &
    (queue["stale_signal"] == 1),
    "reason_code"
] = "STALE_CONTENT"

# Action label
queue["action"] = "REVIEW"

queue.loc[
    queue["reason_code"] == "LOW_CTR",
    "action"
] = "CTR_REVIEW"

queue.loc[
    queue["reason_code"] == "STALE_CONTENT",
    "action"
] = "FRESHNESS_REVIEW"

queue.loc[
    queue["reason_code"] == "LOW_CTR_AND_STALE",
    "action"
] = "PRIORITY_REVIEW"

# Rank the queue
queue = queue.sort_values(
    ["score", "ctr", "days_since_last_update"],
    ascending=[False, True, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

print("Queue size:", len(queue))
print("\nTop 10:")
print(
    queue[
        [
            "rank",
            "content_id",
            "client_id",
            "ctr",
            "days_since_last_update",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10).to_string(index=False)
)

Queue size: 30000

Top 10:
 rank           content_id         client_id  ctr  days_since_last_update  score       reason_code          action
    1 content_55a5b1c46474 client_4ec9599fc2  0.0                     373      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    2 content_f6fdf87348f6 client_4ec9599fc2  0.0                     373      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    3 content_8d56efff1e71 client_4ec9599fc2  0.0                     372      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    4 content_1b4ec72dafd4 client_4ec9599fc2  0.0                     372      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    5 content_e2b702f4f92b client_4ec9599fc2  0.0                     334      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    6 content_06e19c6486b0 client_4ec9599fc2  0.0                     334      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    7 content_7a888d3d99c8 client_19581e27de  0.0                     313      3 LOW_CTR_AND_STALE PRIORITY_REVIEW
    8 content_6476d1d8c050 client_19581e27de  0.0    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I reviewed the top 20 items from the ranked queue.

The highest-ranked items are mainly content with very low CTR and a long time since the last update. These signals make them reasonable candidates for priority review, but the rule is only a decision-support baseline and does not prove that updating the content will improve performance.

For each item, I reviewed the action, reason code, confidence, and what could make the recommendation wrong.

In [11]:
# Review the top 20 ranked items

top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Medium confidence: both low CTR and staleness support review, "
    "but the rule does not prove that an update will improve performance."
)

top20["what_would_make_it_wrong"] = (
    "The low CTR may be caused by low search demand, poor ranking, "
    "or insufficient impressions rather than stale content."
)

review_cols = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_cols]

display(top20_review)

,rank,content_id,client_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_55a5b1c46474,client_4ec9599fc2,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
1,2,content_f6fdf87348f6,client_4ec9599fc2,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
2,3,content_8d56efff1e71,client_4ec9599fc2,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
3,4,content_1b4ec72dafd4,client_4ec9599fc2,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
4,5,content_e2b702f4f92b,client_4ec9599fc2,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
5,6,content_06e19c6486b0,client_4ec9599fc2,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
6,7,content_7a888d3d99c8,client_19581e27de,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
7,8,content_6476d1d8c050,client_19581e27de,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
8,9,content_94991fe6268c,client_19581e27de,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...
9,10,content_02b0d6e30129,client_19581e27de,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW,Medium confidence: both low CTR and staleness ...,The low CTR may be caused by low search demand...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The weakest picks are items where low CTR may not be caused by stale content alone. For example, an item may have low search demand, limited impressions, or a poor ranking position. Therefore, the baseline should be treated as a priority-review queue rather than a guaranteed recommendation.

The rule uses only current snapshot fields: CTR and days since last update. It does not use future-window data, labels, or product-decision flags.

In [12]:
# Inspect weaker / potentially misleading picks

weak_picks = queue[
    (queue["ctr"] == 0) |
    (queue["score"] < queue["score"].max())
].head(10)

display(weak_picks[
    ["rank", "content_id", "client_id", "ctr",
     "days_since_last_update", "score",
     "reason_code", "action"]
])

,rank,content_id,client_id,ctr,days_since_last_update,score,reason_code,action
0,1,content_55a5b1c46474,client_4ec9599fc2,0.0,373,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.0,373,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
2,3,content_8d56efff1e71,client_4ec9599fc2,0.0,372,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
3,4,content_1b4ec72dafd4,client_4ec9599fc2,0.0,372,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
4,5,content_e2b702f4f92b,client_4ec9599fc2,0.0,334,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
5,6,content_06e19c6486b0,client_4ec9599fc2,0.0,334,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
6,7,content_7a888d3d99c8,client_19581e27de,0.0,313,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
7,8,content_6476d1d8c050,client_19581e27de,0.0,313,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
8,9,content_94991fe6268c,client_19581e27de,0.0,313,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW
9,10,content_02b0d6e30129,client_19581e27de,0.0,313,3,LOW_CTR_AND_STALE,PRIORITY_REVIEW


In [13]:
# Leakage check

used_features = ["ctr", "days_since_last_update"]

for col in used_features:
    print(f"{col}: used as a current-snapshot feature")

forbidden_terms = [
    "trend", "label", "future",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

print("\nLeakage check:")
print("The baseline rule uses only current CTR and days_since_last_update.")
print("No label-derived or future-window fields are used.")

ctr: used as a current-snapshot feature
days_since_last_update: used as a current-snapshot feature

Leakage check:
The baseline rule uses only current CTR and days_since_last_update.
No label-derived or future-window fields are used.


All sections are completed.

- The rule uses only current-snapshot signals: CTR and days since last update.
- Two signals were checked with visible bucket tables.
- The ranked queue contains 30,000 items.
- The top 20 items were reviewed individually.
- Weak picks and possible failure cases were documented.
- No label-derived fields or future-window fields were used.
- The baseline is intended for decision-support and priority review, not as a guaranteed action.

In [14]:
import os

output_path = "work/outputs/baseline_action_score.csv"

print("CSV exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size:", os.path.getsize(output_path), "bytes")

CSV exists: False


In [15]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved:", os.path.exists("work/outputs/baseline_action_score.csv"))
print("Rows:", len(queue))

CSV saved: True
Rows: 30000


In [16]:
import os

print(os.listdir("work/outputs"))

['baseline_action_score.csv']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.